In [1]:
# ==========================================================
# Cell 1 : Import Required Libraries
# ==========================================================

import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("="*70)
print("AIS PROLONGED DARK PERIOD ANALYSIS")
print("="*70)
print("Libraries Imported Successfully")

AIS PROLONGED DARK PERIOD ANALYSIS
Libraries Imported Successfully


In [2]:
import pandas as pd

parquet_file = r"C:\Users\developer_02\Desktop\data_AIS\AIS_parquet\ais_destination_cleaned_v3.parquet"

df = pd.read_parquet(parquet_file)

print(df.shape)

(1023554, 23)


In [3]:
# ==========================================================
# Cell 3 : Create Working DataFrame
# ==========================================================

required_columns = [
    "mmsi",
    "voyage_id",
    "vessel_name",
    "timestamp_updated",
    "latitude",
    "longitude"
]

working_df = df[required_columns].copy()

print("="*70)
print("Working DataFrame Created")
print("="*70)

print("Shape :", working_df.shape)

working_df.head()

Working DataFrame Created
Shape : (1023554, 6)


,mmsi,voyage_id,vessel_name,timestamp_updated,latitude,longitude
0,1,VMV0000000001-6b86b273-e3b0c442-NAD-NAT,FISHING NET BOYA-89%,2026-06-01 23:47:32+00:00,24.990640,56.857433
1,1,VMV0000000001-6b86b273-10314348-NAD-NAT,FISHING NET BOYA-88%,2026-06-01 23:57:13+00:00,24.990665,56.857498
2,5,VMV0000000005-ef2d127d-1a03ef9f-NAD-NAT,LONGLINE BUOY 05-69%,2026-06-01 23:10:24+00:00,-32.207008,80.293318
3,5,VMV0000000005-ef2d127d-1a03ef9f-NAD-NAT,LONGLINE BUOY 05-69%,2026-06-01 23:14:20+00:00,-32.207082,80.293187
4,5,VMV0000000005-ef2d127d-cee1aeba-NAD-NAT,JX-7,2026-06-01 23:37:09+00:00,-32.207687,80.292473


In [4]:
# ==========================================================
# Cell 4 : Convert Timestamp
# ==========================================================

working_df["timestamp_updated"] = pd.to_datetime(
    working_df["timestamp_updated"],
    utc=True,
    errors="coerce"
)

print("="*70)
print("Timestamp Conversion Completed")
print("="*70)

print(working_df.dtypes)

Timestamp Conversion Completed
mmsi                               int32
voyage_id                            str
vessel_name                          str
timestamp_updated    datetime64[us, UTC]
latitude                         float64
longitude                        float64
dtype: object


In [5]:
# ==========================================================
# Cell 5 : Dataset Overview
# ==========================================================

print("="*70)
print("Dataset Overview")
print("="*70)

print("\nTotal Rows :", len(working_df))

print("\nUnique MMSI :", working_df["mmsi"].nunique())

print("\nUnique Voyage IDs :", working_df["voyage_id"].nunique())

print("\nUnique Vessel Names :", working_df["vessel_name"].nunique())

print("\nMissing Values\n")

print(working_df.isnull().sum())

Dataset Overview

Total Rows : 1023554

Unique MMSI : 32702

Unique Voyage IDs : 39526

Unique Vessel Names : 33664

Missing Values

mmsi                     0
voyage_id                0
vessel_name          34217
timestamp_updated        0
latitude                 0
longitude                0
dtype: int64


In [6]:
# ==========================================================
# Cell 6 : Unique MMSIs
# ==========================================================

unique_mmsi = working_df["mmsi"].nunique()

print("="*70)
print("Unique MMSI Analysis")
print("="*70)

print(f"Total Unique MMSIs : {unique_mmsi}")

mmsi_counts = (
    working_df["mmsi"]
    .value_counts()
    .rename_axis("mmsi")
    .reset_index(name="AIS Reports")
)

display(mmsi_counts.head(20))

Unique MMSI Analysis
Total Unique MMSIs : 32702


,mmsi,AIS Reports
0,525107017,730
1,525015755,708
2,525019563,704
3,525019511,703
4,525019632,701
5,525401301,700
6,256615000,697
7,525019300,695
8,525019575,689
9,525016601,683


In [7]:
# ==========================================================
# Cell 7 : Unique Voyage IDs
# ==========================================================

unique_voyages = working_df["voyage_id"].nunique()

print("="*70)
print("Unique Voyage ID Analysis")
print("="*70)

print(f"Total Unique Voyage IDs : {unique_voyages}")

voyage_counts = (
    working_df["voyage_id"]
    .value_counts()
    .rename_axis("voyage_id")
    .reset_index(name="AIS Reports")
)

display(voyage_counts.head(20))

Unique Voyage ID Analysis
Total Unique Voyage IDs : 39526


,voyage_id,AIS Reports
0,VMV0525019511-fb797152-51a0f1e9-4e17d2-689,703
1,VIV0009642289-630e7b16-aeeff488-eb7c50-ceb,701
2,VMV0525401301-6a112acb-a0444bbe-7abcf6-a10,700
3,VMV0256615000-9c201087-12acba6e-bf9df6-2ad,697
4,VMV0525019300-d02999ff-0100c18a-7a7bd3-1f1,695
5,VMV0525019575-7ee7e14a-fa5eab26-de6617-742,689
6,VMV0525016601-bcc8844d-a8c83cbd-16e101-751,683
7,VMV0525113040-3656e775-b5d3466a-4f2ddc-24c,683
8,VMV0256062000-124d6a7a-a403bcbe-6fea7f-eb9,677
9,VMV0563036620-50d30a18-ab9ec677-447cf0-NAT,652


In [8]:
# ==========================================================
# Cell 8 : MMSI vs Voyage ID
# ==========================================================

mmsi_voyages = (
    working_df
    .groupby("mmsi")["voyage_id"]
    .nunique()
    .reset_index(name="Unique Voyage IDs")
)

print("="*70)
print("Voyage IDs per MMSI")
print("="*70)

display(mmsi_voyages.head(20))

multiple_voyages = mmsi_voyages[
    mmsi_voyages["Unique Voyage IDs"] > 1
]

print()

print("Vessels having Multiple Voyage IDs :", len(multiple_voyages))

display(multiple_voyages.head(20))

Voyage IDs per MMSI


,mmsi,Unique Voyage IDs
0,1,2
1,5,3
2,10,3
3,11,2
4,16,1
5,36,1
6,41,1
7,197,1
8,296,1
9,492,1



Vessels having Multiple Voyage IDs : 5186


,mmsi,Unique Voyage IDs
0,1,2
1,5,3
2,10,3
3,11,2
10,798,2
13,886,3
14,991,2
16,1111,2
18,1512,2
26,9443,2


In [9]:
# ==========================================================
# Cell 9 : Voyage ID vs MMSI
# ==========================================================

voyage_mmsi = (
    working_df
    .groupby("voyage_id")["mmsi"]
    .nunique()
    .reset_index(name="Unique MMSIs")
)

print("="*70)
print("MMSIs per Voyage ID")
print("="*70)

display(voyage_mmsi.head(20))

shared_voyages = voyage_mmsi[
    voyage_mmsi["Unique MMSIs"] > 1
]

print()

print("Voyage IDs Shared by Multiple MMSIs :", len(shared_voyages))

display(shared_voyages.head(20))

MMSIs per Voyage ID


,voyage_id,Unique MMSIs
0,VIV0001006805-943530cd-2dfa4c38-384397-399,1
1,VIV0001006805-943530cd-2dfa4c38-43753a-399,1
2,VIV0001008592-7aa5583b-45da7fc5-8521b2-647,1
3,VIV0001008592-7aa5583b-45da7fc5-e8150c-647,1
4,VIV0001011331-7653e3dc-aec6a9e0-ff4def-bf5,1
5,VIV0001011642-948b694f-8e404d32-a724ee-03e,1
6,VIV0001012713-cae54b58-5a190879-f8fd65-148,1
7,VIV0001013145-d84a212b-4460c5cc-1edab2-ac0,1
8,VIV0001013640-746530b7-f2bf3840-59b0dd-dbf,1
9,VIV0001013767-07592f2d-f69935dd-b889f4-ce5,1



Voyage IDs Shared by Multiple MMSIs : 0


,voyage_id,Unique MMSIs


In [10]:
# ==========================================================
# Cell 10 : Decide Grouping Strategy
# ==========================================================

total_mmsi = working_df["mmsi"].nunique()

total_voyages = working_df["voyage_id"].nunique()

multiple_voyage_vessels = (
    working_df
    .groupby("mmsi")["voyage_id"]
    .nunique()
    .gt(1)
    .sum()
)

shared_voyages = (
    working_df
    .groupby("voyage_id")["mmsi"]
    .nunique()
    .gt(1)
    .sum()
)

print("="*70)
print("Grouping Strategy Analysis")
print("="*70)

print(f"Unique MMSIs                     : {total_mmsi}")
print(f"Unique Voyage IDs               : {total_voyages}")
print(f"MMSIs with Multiple Voyage IDs  : {multiple_voyage_vessels}")
print(f"Voyage IDs shared by MMSIs      : {shared_voyages}")

print("\nRecommended Grouping Strategy:")

if multiple_voyage_vessels > 0:
    print("➡ Group by ['mmsi', 'voyage_id']")
else:
    print("➡ Group by ['mmsi'] only")

Grouping Strategy Analysis
Unique MMSIs                     : 32702
Unique Voyage IDs               : 39526
MMSIs with Multiple Voyage IDs  : 5186
Voyage IDs shared by MMSIs      : 0

Recommended Grouping Strategy:
➡ Group by ['mmsi', 'voyage_id']


In [11]:
# ==========================================================
# Cell 11 : Sort Data & Set Dark Threshold
# ==========================================================

print("="*80)
print("CELL 11 : SORTING DATA")
print("="*80)

# ----------------------------------------------------------
# Dark Period Threshold
# ----------------------------------------------------------

DARK_THRESHOLD_HOURS = 4

print(f"Dark Period Threshold : {DARK_THRESHOLD_HOURS} Hours")

# ----------------------------------------------------------
# Sort Chronologically
# ----------------------------------------------------------

working_df = working_df.sort_values(
    by=[
        "mmsi",
        "voyage_id",
        "timestamp_updated"
    ]
).reset_index(drop=True)

print("\nData Sorted Successfully")

print("\nFirst 5 Records")

display(working_df.head())

CELL 11 : SORTING DATA
Dark Period Threshold : 4 Hours

Data Sorted Successfully

First 5 Records


,mmsi,voyage_id,vessel_name,timestamp_updated,latitude,longitude
0,1,VMV0000000001-6b86b273-10314348-NAD-NAT,FISHING NET BOYA-88%,2026-06-01 23:57:13+00:00,24.990665,56.857498
1,1,VMV0000000001-6b86b273-e3b0c442-NAD-NAT,FISHING NET BOYA-89%,2026-06-01 23:47:32+00:00,24.990640,56.857433
2,5,VMV0000000005-ef2d127d-1a03ef9f-NAD-NAT,LONGLINE BUOY 05-69%,2026-06-01 23:10:24+00:00,-32.207008,80.293318
3,5,VMV0000000005-ef2d127d-1a03ef9f-NAD-NAT,LONGLINE BUOY 05-69%,2026-06-01 23:14:20+00:00,-32.207082,80.293187
4,5,VMV0000000005-ef2d127d-844eff20-NAD-NAT,KAWATEA5,2026-06-01 23:40:11+00:00,-32.207837,80.292285


In [12]:
# ==========================================================
# Cell 12 : Previous AIS Information
# ==========================================================

print("="*80)
print("CELL 12 : PREVIOUS AIS REPORT")
print("="*80)

group_cols = ["mmsi", "voyage_id"]

working_df["previous_timestamp"] = (
    working_df
    .groupby(group_cols)["timestamp_updated"]
    .shift(1)
)

working_df["previous_latitude"] = (
    working_df
    .groupby(group_cols)["latitude"]
    .shift(1)
)

working_df["previous_longitude"] = (
    working_df
    .groupby(group_cols)["longitude"]
    .shift(1)
)

print("Previous AIS Information Created")

display(
    working_df[
        [
            "mmsi",
            "voyage_id",
            "timestamp_updated",
            "previous_timestamp",
            "latitude",
            "previous_latitude",
            "longitude",
            "previous_longitude"
        ]
    ].head(10)
)

CELL 12 : PREVIOUS AIS REPORT
Previous AIS Information Created


,mmsi,voyage_id,timestamp_updated,previous_timestamp,latitude,previous_latitude,longitude,previous_longitude
0,1,VMV0000000001-6b86b273-10314348-NAD-NAT,2026-06-01 23:57:13+00:00,NaT,24.990665,NaN,56.857498,NaN
1,1,VMV0000000001-6b86b273-e3b0c442-NAD-NAT,2026-06-01 23:47:32+00:00,NaT,24.990640,NaN,56.857433,NaN
2,5,VMV0000000005-ef2d127d-1a03ef9f-NAD-NAT,2026-06-01 23:10:24+00:00,NaT,-32.207008,NaN,80.293318,NaN
3,5,VMV0000000005-ef2d127d-1a03ef9f-NAD-NAT,2026-06-01 23:14:20+00:00,2026-06-01 23:10:24+00:00,-32.207082,-32.207008,80.293187,80.293318
4,5,VMV0000000005-ef2d127d-844eff20-NAD-NAT,2026-06-01 23:40:11+00:00,NaT,-32.207837,NaN,80.292285,NaN
5,5,VMV0000000005-ef2d127d-cee1aeba-NAD-NAT,2026-06-01 23:37:09+00:00,NaT,-32.207687,NaN,80.292473,NaN
6,5,VMV0000000005-ef2d127d-cee1aeba-NAD-NAT,2026-06-01 23:42:11+00:00,2026-06-01 23:37:09+00:00,-32.207797,-32.207687,80.292350,80.292473
7,5,VMV0000000005-ef2d127d-cee1aeba-NAD-NAT,2026-06-01 23:47:13+00:00,2026-06-01 23:42:11+00:00,-32.207753,-32.207797,80.292463,80.292350
8,5,VMV0000000005-ef2d127d-cee1aeba-NAD-NAT,2026-06-01 23:51:32+00:00,2026-06-01 23:47:13+00:00,-32.207580,-32.207753,80.292450,80.292463
9,10,VMV0000000010-4a44dc15-a9d9832b-NAD-NAT,2026-06-01 23:58:12+00:00,NaT,-34.890792,NaN,101.508593,NaN


In [13]:
# ==========================================================
# Cell 13 : Calculate Time Gap
# ==========================================================

print("="*80)
print("CELL 13 : TIME GAP CALCULATION")
print("="*80)

working_df["time_gap"] = (
    working_df["timestamp_updated"]
    -
    working_df["previous_timestamp"]
)

working_df["gap_seconds"] = (
    working_df["time_gap"]
    .dt.total_seconds()
)

working_df["gap_minutes"] = (
    working_df["gap_seconds"] / 60
)

working_df["gap_hours"] = (
    working_df["gap_seconds"] / 3600
)

print("Gap Calculation Completed")

display(
    working_df[
        [
            "mmsi",
            "timestamp_updated",
            "previous_timestamp",
            "gap_hours"
        ]
    ].head(10)
)

CELL 13 : TIME GAP CALCULATION
Gap Calculation Completed


,mmsi,timestamp_updated,previous_timestamp,gap_hours
0,1,2026-06-01 23:57:13+00:00,NaT,NaN
1,1,2026-06-01 23:47:32+00:00,NaT,NaN
2,5,2026-06-01 23:10:24+00:00,NaT,NaN
3,5,2026-06-01 23:14:20+00:00,2026-06-01 23:10:24+00:00,0.065556
4,5,2026-06-01 23:40:11+00:00,NaT,NaN
5,5,2026-06-01 23:37:09+00:00,NaT,NaN
6,5,2026-06-01 23:42:11+00:00,2026-06-01 23:37:09+00:00,0.083889
7,5,2026-06-01 23:47:13+00:00,2026-06-01 23:42:11+00:00,0.083889
8,5,2026-06-01 23:51:32+00:00,2026-06-01 23:47:13+00:00,0.071944
9,10,2026-06-01 23:58:12+00:00,NaT,NaN
